In [1]:
import json

# --- Load the Surya handoff from notebook 11 (or re-run Surya if needed) ---
# For the PoC we reuse the saved OCR output: sentences + per-line confidence.
with open("../data/notebook11_handoff.json", "r", encoding="utf-8") as f:
    handoff = json.load(f)

# The handoff currently has document-level data. For real routing we want
# PER-LINE confidence paired with PER-LINE text. Note for later: notebook 11
# saved sentence-level reassembled text but LINE-level confidence — these
# don't align 1:1 because reassembly merges lines. For the PoC we route at
# the granularity we have; flag this as a known limitation to fix.

sentences = handoff["sentences"]
line_confidences = handoff["line_confidences"]

print(f"{len(sentences)} sentences, {len(line_confidences)} line confidences")
print(f"Min confidence: {min(line_confidences):.4f}")

5 sentences, 8 line confidences
Min confidence: 0.9569


In [2]:
# === Document-level routing logic ===
# Decision rule from your calibration finding (notebook 07/08):
#   min line confidence >= 0.92  -> TRUST: auto-translate
#   min line confidence <  0.92  -> FLAG: route to human interpreter
CONFIDENCE_THRESHOLD = 0.92

doc_min_confidence = min(line_confidences)
route = "AUTO_TRANSLATE" if doc_min_confidence >= CONFIDENCE_THRESHOLD else "FLAG_FOR_INTERPRETER"

print("=== ROUTING DECISION ===")
print(f"Document min confidence: {doc_min_confidence:.4f}")
print(f"Threshold:               {CONFIDENCE_THRESHOLD}")
print(f"Decision:                {route}")
print()

if route == "AUTO_TRANSLATE":
    print("-> OCR confidence sufficient. Proceeding to translation.")
    print("-> Output will be shown to officer WITH an 'AI Translation' label")
    print("   and the confidence score (EU AI Act Art. 13 transparency).")
else:
    print("-> OCR confidence too low. Translation withheld from officer.")
    print("-> Officer is shown: 'Document quality insufficient for reliable")
    print("   translation. Connect to a human interpreter?' (Art. 14 oversight)")
    print("-> The 'connect to interpreter' button (Amic's Art.14 point) is the")
    print("   escalation path here.")

=== ROUTING DECISION ===
Document min confidence: 0.9569
Threshold:               0.92
Decision:                AUTO_TRANSLATE

-> OCR confidence sufficient. Proceeding to translation.
-> Output will be shown to officer WITH an 'AI Translation' label
   and the confidence score (EU AI Act Art. 13 transparency).


In [3]:
# === Validate the router on known cases from KITAB-Bench (notebook 07) ===
# Real samples: (sample_id, min_confidence, actual_CER) from your calibration run
test_cases = [
    ("This document (clean Wikipedia)", doc_min_confidence, None),
    ("KITAB sample 43 (hallucinated repetition)", 0.3295, 2.8824),
    ("KITAB sample 34 (recognition collapse)",    0.6235, 1.0000),
    ("KITAB sample 12 (partial collapse)",         0.4755, 0.7500),
    ("KITAB sample 14 (Latin contamination)",      0.1715, 0.6024),
    ("KITAB sample 13 (the uncatchable one)",      0.9972, 0.1184),
    ("KITAB sample 2 (clean)",                     0.9953, 0.0000),
]

print("=== ROUTER VALIDATION ===\n")
print(f"{'Case':<45}{'Conf':<8}{'Route':<22}{'Correct?'}")
print("-" * 90)

for name, conf, cer in test_cases:
    route = "AUTO_TRANSLATE" if conf >= CONFIDENCE_THRESHOLD else "FLAG_FOR_INTERPRETER"
    # "Correct" = flagged the ones that were actually bad (CER >= 10%)
    if cer is None:
        correct = "-"
    else:
        actually_bad = cer >= 0.10
        flagged = (route == "FLAG_FOR_INTERPRETER")
        correct = "YES" if (actually_bad == flagged) else "NO (!!)"
    print(f"{name:<45}{conf:<8.4f}{route:<22}{correct}")

=== ROUTER VALIDATION ===

Case                                         Conf    Route                 Correct?
------------------------------------------------------------------------------------------
This document (clean Wikipedia)              0.9569  AUTO_TRANSLATE        -
KITAB sample 43 (hallucinated repetition)    0.3295  FLAG_FOR_INTERPRETER  YES
KITAB sample 34 (recognition collapse)       0.6235  FLAG_FOR_INTERPRETER  YES
KITAB sample 12 (partial collapse)           0.4755  FLAG_FOR_INTERPRETER  YES
KITAB sample 14 (Latin contamination)        0.1715  FLAG_FOR_INTERPRETER  YES
KITAB sample 13 (the uncatchable one)        0.9972  AUTO_TRANSLATE        NO (!!)
KITAB sample 2 (clean)                       0.9953  AUTO_TRANSLATE        YES
